# Data harmonisation

The `unmapped entity` will be converted into `linked_eneties` with the link with other datasets (i.e. secondary, supplementary, and linked_entity).

The harmonisation process:
1. read the unmapped entities, one by one
2. link the unmapped entity one by one, by linking each key property (foreign id); when link to the dataset, check the existing data (using LLM/ollama)
    - if there are existing data (e.g. technology), then use the existing one
    - if no existing data, then create a few on
3. create a linked entity form the unmapped entity, fill with foreign key

### Import Libraries
This cell imports the necessary Python libraries for data manipulation and working with YAML files.

In [1]:
import os
import pandas as pd
import yaml
from pathlib import Path
import csv
import json

In [2]:
import importlib
import harmonise_helpers as hh
importlib.reload(hh)  # pick up edits without restarting the kernel

from harmonise_helpers import (
    MODEL, ENTITY_CONFIG, SCOPE_CONFIG, ATTR_PATH, ATTR_COLS, LE_PATH, MAPPING_DIR,
    load_registry, append_row, load_attr_registry,
    llm_fill_fields, validate_row,
    resolve_entity, ensure_attr, ensure_scope,
    collect_candidates, generate_audit,
    backup_derived_data, reset_derived_data,
)

print("Helpers loaded from harmonise_helpers.py")

Helpers loaded from harmonise_helpers.py


### Load Controlled Vocabulary Context for LLM
Loads the **Nomenclature** (term definitions for all CV fields) and **Carrier** (valid carrier abbreviations) 
from the source Excel and injects them into the LLM system prompt via `hh.GLOBAL_CV_CONTEXT`.
This gives the LLM full terminology context when filling fields for attributes, scopes, carriers, and other CV types.

In [ ]:
_refuel_excel = Path("../1_ingest/ingestion_space/refuel/raw_data/reFuel_TechDatabase_Clean_2026-06-03.xlsx")
_refuel_sheets = pd.read_excel(_refuel_excel, sheet_name=["Nomenclature", "Carrier"])

def _build_cv_context(df_nom, df_car):
    lines = ["The following controlled vocabulary definitions apply to all fields in this database."]

    lines.append("
--- Nomenclature: term definitions used across all controlled vocabulary fields ---")
    for _, row in df_nom.iterrows():
        term = row.get("Term / Abbreviation")
        defn = row.get("Definition")
        if pd.notna(term) and pd.notna(defn) and str(term).strip():
            lines.append(f"{term}: {defn}")

    lines.append("
--- Carrier: valid abbreviations for all carrier fields ---")
    for _, row in df_car.iterrows():
        abbr = row.get("Carrier Abbreviation")
        desc = row.get("Carrier Description")
        ctype = row.get("Carrier Type")
        if pd.notna(abbr) and pd.notna(desc) and str(abbr).strip():
            tag = f" ({ctype})" if pd.notna(ctype) else ""
            lines.append(f"{abbr}{tag}: {desc}")

    return "
".join(lines)

hh.GLOBAL_CV_CONTEXT = _build_cv_context(_refuel_sheets["Nomenclature"], _refuel_sheets["Carrier"])
print(f"CV context loaded ({len(hh.GLOBAL_CV_CONTEXT)} chars, {len(hh.GLOBAL_CV_CONTEXT.splitlines())} lines)")

## Load data

### Load All Schema Definitions
This cell defines a function to recursively load all YAML schema files from the `../schema/` directory into a dictionary called `all_schemas`. These schemas are crucial for understanding the structure and validation rules for different data entities.

In [3]:
# Define the base directory for schema files
base_dir = '../schema/'

# Initialize an empty dictionary to hold all loaded schemas
all_schemas = {}

# Function to recursively load YAML files from a given directory
def load_yaml_files(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.yaml'):
                file_path = os.path.join(root, file)
                with open(file_path, 'r', encoding='utf-8') as f:
                    schema = yaml.safe_load(f)
                    all_schemas[file] = schema

# Call the function to load all YAML files from the schema directory and its subdirectories
load_yaml_files(base_dir)

# Print the number of loaded schemas for verification
print(f"Loaded {len(all_schemas)} schema files.")

Loaded 13 schema files.


### Load Unmapped Entities
This cell loads the unmapped entities from the `unmapped_entities_refuel.yaml` file, which will be used for harmonisation.

In [4]:
## load the unmapped entities from the yaml file
path = r'../motel-db\unmapped_entity\unmapped_entities_refuel.yaml'

with open(path, "r", encoding="utf-8") as f:
    ue = yaml.safe_load(f)

print(f"Unmapped entities: {len(ue)}")

Unmapped entities: 99


In [5]:
# test on the first 3 unmapped entities
ue = ue[:]

### Load Existing Datasets
This cell defines a function to recursively load all CSV data files from the `../motel-db/` directory into a dictionary called `all_csv_data`. These datasets represent existing linked entities and controlled vocabularies that will be used during the harmonisation process.

In [6]:
# --- Optional: backup then reset all derived data before a fresh run ---
# backup_derived_data() copies current files to motel-db/_backup/<timestamp>/
# reset_derived_data() wipes and recreates each file with schema-correct headers.
# Run both in sequence to preserve the previous state before reprocessing.

backup_derived_data()
reset_derived_data()

=== Backup saved to ..\motel-db\_backup\20260623_192958 ===
Copied:
  - ..\motel-db\controlled_vocabulary\attribute.csv
  - ..\motel-db\controlled_vocabulary\geographic_scope.csv
  - ..\motel-db\controlled_vocabulary\temporal_scope.csv
  - ..\motel-db\controlled_vocabulary\capacity_scope.csv
  - ..\motel-db\controlled_vocabulary\system_boundary.csv
  - ..\motel-db\secondary\technology.csv
  - ..\motel-db\secondary\process.csv
  - ..\motel-db\secondary\source.csv
  - ..\motel-db\controlled_vocabulary\carrier.csv
  - ..\motel-db\supplementary\contributor.csv
  - ..\motel-db\supplementary\review.csv
  - ..\motel-db\mapping/ (directory)
=== Reset complete ===
  [reset] ..\motel-db\controlled_vocabulary\attribute.csv
           columns: attribute_id, attribute_name, attribute_description, unit, data_format, ontology_iri, note
  [reset] ..\motel-db\controlled_vocabulary\geographic_scope.csv
           columns: geographic_scope, geographic_scope_description, note
  [reset] ..\motel-db\control

In [7]:
# Define the base directory for data files
data_dir = '../motel-db/'

# Initialize an empty dictionary to hold all loaded CSV data
all_csv_data = {}

# Function to recursively load CSV files from a given directory
def load_csv_files(directory):
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.csv'):
                file_path = os.path.join(root, file)
                with open(file_path, mode='r', encoding='utf-8') as f:
                    try:
                        data = pd.read_csv(f)
                    except Exception as e:
                        print(f"Error reading {file_path}: {e}")
                        data = None
                        continue
                    # Use the filename (without path/extension) or full path as key
                    file_name = file.split('.')[0]  # or use file_path for full paSth
                    all_csv_data[file_name] = data

# Load all CSV files from the motel-db directory and its subdirectories
load_csv_files(data_dir)

# Print the number of loaded CSV files for verification
print(f"Loaded {len(all_csv_data)} CSV files.")

Loaded 21 CSV files.


# Harmonisation process

### Harmonisation Configuration and Helper Functions
All configuration (entity types, file paths, column definitions) and helper functions
are defined in [harmonise_helpers.py](harmonise_helpers.py) and imported here.

The module provides:
- `ENTITY_CONFIG`, `SCOPE_CONFIG`, `ATTR_PATH`, `LE_PATH`, `MAPPING_DIR` — path and schema config
- `load_registry`, `append_row`, `load_attr_registry` — CSV registry I/O
- `llm_fill_fields`, `validate_row` — schema-guided LLM field filling and validation
- `resolve_entity` — three-stage resolver (exact → LLM → create)
- `ensure_attr`, `ensure_scope` — controlled vocabulary helpers
- `collect_candidates` — deduplication of unmapped entity candidates
- `generate_audit` — per-entity audit report joining all mapping files
- `reset_derived_data` — deletes all derived files (linked entities, attributes, scopes, mappings) while leaving source registries intact

In [8]:
# all_schemas is already loaded above — make it available to resolve_entity via the module
# (pass it explicitly when calling resolve_entity)
ATTR_SCHEMA = all_schemas.get("attribute.yaml", {})
print(f"Schema keys available: {list(all_schemas.keys())}")

Schema keys available: ['linked_entity.yaml', 'unmapped_entity.yaml', 'attribute.yaml', 'capacity_scope.yaml', 'carrier.yaml', 'geographic_scope.yaml', 'system_boundary.yaml', 'temporal_scope.yaml', 'process.yaml', 'source.yaml', 'technology.yaml', 'contributor.yaml', 'review.yaml']


### Step 1: Collect Unique Candidates
This cell iterates through the unmapped entities and extracts unique candidates for technology, process, source, and carrier types. This ensures that each unique name is only resolved once, improving efficiency and consistency.

In [9]:
# --- Step 1: Collect unique candidates per entity type (dedup so each name is resolved once) ---

def collect_candidates(unmapped_entities):
    """
    Extracts unique entity candidates from a list of unmapped entities.

    Iterates over each unmapped entity and collects distinct candidates for
    technology, process, source, and carrier types. Deduplication is done by
    entity name so each unique name is resolved only once in subsequent steps.

    Args:
        unmapped_entities (list[dict]): List of unmapped entity dicts, each containing
            fields like "technology_name", "technology", "sources", and "balancing".

    Returns:
        dict[str, dict]: A mapping from entity type (e.g. "technology", "process",
            "source", "carrier") to a dict of {name: candidate_dict}, where each
            candidate_dict holds the fields needed for registry resolution.
    """
    candidates = {et: {} for et in ENTITY_CONFIG}
    for e in unmapped_entities:
        t = e.get("technology", {})
        tname = e.get("technology_name")
        if tname:
            candidates["technology"].setdefault(tname, {
                "technology_name": tname,
                "technology_type": t.get("technology_type"),
                "technology_category": t.get("technology_category"),
                "technology_description": t.get("technology_description"),
            })
        pname = t.get("process_name")
        if pname:
            candidates["process"].setdefault(pname, {
                "process_name": pname,
                "process_type": t.get("process_type"),
                "process_category": t.get("process_category"),
            })
        for src in e.get("sources", []):
            sname = src.get("source_name")
            if sname:
                candidates["source"].setdefault(sname, {
                    "source_name": sname,
                    "source_description": src.get("source_description"),
                })
        for item in e.get("balancing", {}).get("inputs", []) + e.get("balancing", {}).get("outputs", []):
            cname = item.get("carrier_name")
            if cname:
                candidates["carrier"].setdefault(cname, {"carrier_name": cname})
    return candidates

candidates = collect_candidates(ue)
for et, items in candidates.items():
    print(f"  {et}: {len(items)} unique candidates")

  technology: 57 unique candidates
  process: 39 unique candidates
  source: 25 unique candidates
  carrier: 37 unique candidates


### Step 2: Resolve Entities via LLM
For each unique candidate collected in Step 1, this cell resolves it against the existing registry using `resolve_entity`. Resolution proceeds in two stages:

1. **Exact match** — the candidate name is compared case-insensitively against all registry entries; if a match is found the existing ID is reused immediately with no LLM call.
2. **Semantic match (LLM)** — if no exact match exists and the registry is non-empty, the full registry and candidate are sent to the local Ollama model, which returns a JSON verdict indicating whether a semantically equivalent entry already exists and, if so, its ID.

If neither stage finds a match, a new row is created with a freshly generated ID, appended to the CSV, and added to the in-memory registry so subsequent candidates in the same run can match against it.

In [10]:
# --- Step 2: Resolve all LLM-backed entities ---
registries = {et: load_registry(et) for et in ENTITY_CONFIG}
resolved_ids = {et: {} for et in ENTITY_CONFIG}        # {entity_type: {name -> id}}
resolution_status = {et: {} for et in ENTITY_CONFIG}   # {entity_type: {name -> status}}

for entity_type, entity_candidates in candidates.items():
    print(f"\nResolving {entity_type}...")
    registry = registries[entity_type]
    name_field = ENTITY_CONFIG[entity_type]["name_field"]
    counts = {"exact": 0, "llm": 0, "created": 0}
    for name, candidate in entity_candidates.items():
        rid, status = resolve_entity(entity_type, candidate, registry, all_schemas)
        resolved_ids[entity_type][name] = rid
        resolution_status[entity_type][name] = status
        counts[status] += 1
    total = sum(counts.values())
    print(f"  total: {total}  |  exact match: {counts['exact']}  |  LLM match: {counts['llm']}  |  newly created: {counts['created']}")

print("\nLLM resolution complete.")


Resolving technology...
  + technology: TECH_00001  (NH3_CCGT_El)
  + technology: TECH_00002  (NH3_OCGT_El)
  + technology: TECH_00003  (CH4_CCGT_El)
  + technology: TECH_00004  (CH4_CHP_ElTh)
  + technology: TECH_00005  (CH4_FuelCell_El)
  + technology: TECH_00006  (CH4_OCGT_El)
  + technology: TECH_00007  (Geothermal_CHP_ElTh)
  + technology: TECH_00008  (H2_CCGT_El)
  + technology: TECH_00009  (H2_FuelCell_El)
  + technology: TECH_00010  (H2_FuelCell_ElBdg)
  + technology: TECH_00011  (H2_OCGT_El)
  + technology: TECH_00012  (El_PumpedHydro_Pump)
  + technology: TECH_00013  (Hydro_PumpedTurbine_El)
  + technology: TECH_00014  (Hydro_RunOfRiver_El)
  + technology: TECH_00015  (Hydro_Reservoir_El)
  + technology: TECH_00016  (MSW_Incineration_El)
  + technology: TECH_00017  (Solar_PVAlpine_El)
  + technology: TECH_00018  (Solar_PVRooftop_El)
  + technology: TECH_00019  (U235_Nuclear_El)
  + technology: TECH_00020  (Wind_Turbine_El)
  + technology: TECH_00021  (Wood_CHP_ElTh)
  + tech

### Step 3: Resolve Controlled Vocabulary (Attributes & Scopes)
This step handles entities whose values come from fixed controlled vocabularies — **attributes** and **scopes** (geographic, temporal, capacity, system boundary). Unlike Steps 1–2, resolution here is fully deterministic: no LLM is used.

- **Attributes** (`ensure_attr`) — looks up the attribute name in the in-memory registry; if absent, generates a new `ATTR_XXXXX` ID and appends a row to `attribute.csv`.
- **Scopes** (`ensure_scope`) — checks whether the scope token already exists in the corresponding CSV; if not, appends it. The token itself (e.g. `"ECA"`, `"2050"`) is used as the foreign key in linked entities.

Both helpers are idempotent: re-running the cell with the same data will not create duplicate entries.

In [11]:
# --- Step 3: Resolve controlled vocabulary (attributes + scopes) ---
ATTR_SCHEMA = all_schemas.get("attribute.yaml", {})

# Repair: rebuild attribute.csv from scratch (delete corrupted file first)
print("Rebuilding attribute.csv from scratch...")
Path(ATTR_PATH).unlink(missing_ok=True)

attr_registry = {}
attr_ids   = {}
scope_ids  = {}
attr_counts  = {"existing": 0, "created": 0}
scope_counts = {"existing": 0, "created": 0}

# Rebuild attributes from all 99 unmapped entities so the registry is complete
with open(r'../motel-db/unmapped_entity/unmapped_entities_refuel.yaml', "r", encoding="utf-8") as f:
    ue_all = yaml.safe_load(f)

for entity in ue_all:
    for attr in entity.get("attributes", []):
        name = attr.get("attribute_name")
        if name and name not in attr_ids:
            aid, status = ensure_attr(name, registry=attr_registry, notes=attr.get("notes", ""), attr_schema=ATTR_SCHEMA)
            attr_ids[name] = aid
            attr_counts[status] += 1

# Resolve scopes for the working slice only
for entity in ue:
    scope = entity.get("scope", {})
    for scope_type in SCOPE_CONFIG:
        val = scope.get(f"{scope_type}_description")
        key = (scope_type, val)
        if val and key not in scope_ids:
            token, status = ensure_scope(scope_type, val)
            scope_ids[key] = token
            if status:
                scope_counts[status] += 1

print(f"Attributes   — total: {sum(attr_counts.values())}  |  existing: {attr_counts['existing']}  |  created: {attr_counts['created']}")
print(f"Scope tokens — total: {sum(scope_counts.values())}  |  existing: {scope_counts['existing']}  |  created: {scope_counts['created']}")

Rebuilding attribute.csv from scratch...
  [WARN] attribute:tech_maturity — missing required fields after fill: ['unit']
Attributes   — total: 15  |  existing: 0  |  created: 15
Scope tokens — total: 15  |  existing: 0  |  created: 15


### Step 4: Build Linked Entity Records and Save
With all foreign keys resolved (technology, process, source, carrier, attributes, scopes), this step assembles the final `linked_entity` records and writes them to `linked_entity/linked_entity.yaml`.

YAML is used instead of CSV because the linked entity schema is deeply nested — sources, balancing flows, and attribute values are arrays of objects that do not flatten cleanly into tabular columns.

For each unmapped entity:
- **Foreign keys** for technology and process are looked up from `resolved_ids` (Step 2).
- **Scope tokens** are looked up from `scope_ids` (Step 3).
- **Balancing flows** (inputs/outputs) are structured as `{carrier_id, share, unit}` objects.
- **Sources** are structured as `{source_id, linked_attribute_ids}` objects.
- **Attribute values** are structured as `{attribute_id, attribute_name, value, time_index}` objects.

The output file is overwritten on every run.

In [12]:
# --- Step 4: Build linked_entity records and save as YAML ---
from datetime import date

linked_entities = []
today = str(date.today())

for i, entity in enumerate(ue):
    t     = entity.get("technology", {})
    scope = entity.get("scope", {})

    linked_entities.append({
        "linked_entity_id": f"LE_{i+1:05d}",
        "tech_id":     resolved_ids["technology"].get(entity.get("technology_name"), ""),
        "process_id":  resolved_ids["process"].get(t.get("process_name"), ""),
        "scope": {
            "geographic_scope": scope_ids.get(("geographic_scope", scope.get("geographic_scope_description")), ""),
            "temporal_scope":   scope_ids.get(("temporal_scope",   scope.get("temporal_scope_description")), ""),
            "capacity_scope":   scope_ids.get(("capacity_scope",   scope.get("capacity_scope_description")), ""),
            "system_boundary":  scope_ids.get(("system_boundary",  scope.get("system_boundary_description")), ""),
        },
        "balancing": {
            "inputs":  [
                {"carrier_id": resolved_ids["carrier"].get(x["carrier_name"], ""), "share": x.get("share"), "unit": x.get("unit")}
                for x in entity.get("balancing", {}).get("inputs", []) if x.get("carrier_name")
            ],
            "outputs": [
                {"carrier_id": resolved_ids["carrier"].get(x["carrier_name"], ""), "share": x.get("share"), "unit": x.get("unit")}
                for x in entity.get("balancing", {}).get("outputs", []) if x.get("carrier_name")
            ],
        },
        "sources": [
            {
                "source_id": resolved_ids["source"].get(s["source_name"], ""),
                "linked_attribute_ids": [
                    attr_ids.get(a) or f"[unregistered: {a}]"
                    for a in s.get("linked_attribute", [])
                ],
            }
            for s in entity.get("sources", []) if s.get("source_name")
        ],
        "values": [
            {
                "attribute_id":   attr_ids.get(a["attribute_name"], ""),
                "attribute_name": a["attribute_name"],
                "value":          a.get("value"),
                "time_index":     a.get("time_index"),
            }
            for a in entity.get("attributes", []) if a.get("attribute_name")
        ],
        "date_created": today,
    })

Path(LE_PATH).parent.mkdir(parents=True, exist_ok=True)
with open(LE_PATH, "w", encoding="utf-8") as f:
    yaml.dump(linked_entities, f, allow_unicode=True, sort_keys=False)

print(f"Saved {len(linked_entities)} linked entities -> {LE_PATH}")
for le in linked_entities:
    print(f"  {le['linked_entity_id']}  tech={le['tech_id']}  process={le['process_id']}  scope={le['scope']}")

Saved 99 linked entities -> ../motel-db/linked_entity/linked_entity.yaml
  LE_00001  tech=TECH_00001  process=PROC_00001  scope={'geographic_scope': 'ECA', 'temporal_scope': '2050', 'capacity_scope': '', 'system_boundary': 'Plant ready to operate'}
  LE_00002  tech=TECH_00002  process=PROC_00002  scope={'geographic_scope': 'ECA', 'temporal_scope': '2050', 'capacity_scope': '', 'system_boundary': 'Plant ready to operate'}
  LE_00003  tech=TECH_00003  process=PROC_00001  scope={'geographic_scope': 'ECA', 'temporal_scope': '2050', 'capacity_scope': '400000', 'system_boundary': 'Plant ready to operate'}
  LE_00004  tech=TECH_00004  process=PROC_00003  scope={'geographic_scope': 'ECA', 'temporal_scope': '2050', 'capacity_scope': '1000', 'system_boundary': 'Plant ready to operate'}
  LE_00005  tech=TECH_00005  process=PROC_00004  scope={'geographic_scope': 'ECA', 'temporal_scope': '2030', 'capacity_scope': '', 'system_boundary': 'Plant ready to operate'}
  LE_00006  tech=TECH_00005  process=

### Step 5: Save Mapping Files and Audit
After harmonisation, two types of mapping files are written to `motel-db/mapping/` to record how every input name was resolved. These files are useful for auditing, re-running, and tracing data lineage.

**Provenance map** (`unmapped_to_linked.csv`) — one row per unmapped entity, linking its original index and technology name to the resulting `linked_entity_id` and all resolved foreign keys (tech, process, scope, sources). This is the primary traceability record: given any linked entity, you can trace it back to the exact unmapped input.

**Entity lookup maps** (one CSV per entity type: `technology_map.csv`, `process_map.csv`, `source_map.csv`, `carrier_map.csv`, `attribute_map.csv`, plus one per scope type) — each maps an original name or value to its assigned ID and a `status` field (`exact`, `llm`, `created`, or `pre-existing`). These show at a glance which entities were reused from the existing registry and which were newly introduced during this run.

**Per-entity audit report** — after the mapping files are saved, a per-entity audit joins all resolved mappings and prints a structured summary for each entity: assigned IDs, resolution status for technology, process, sources, carriers, and scope, plus any `unresolved_attributes`. Use this to spot-check the harmonisation output before committing results.

In [13]:
# --- Step 5: Save mapping files ---
MAPPING_DIR.mkdir(parents=True, exist_ok=True)

# --- Provenance map: one row per unmapped entity -> linked entity ---
MAP_A_PATH = MAPPING_DIR / "unmapped_to_linked.csv"
MAP_A_COLS = [
    "unmapped_index", "technology_name",
    "linked_entity_id", "tech_id", "process_id",
    "geographic_scope", "temporal_scope", "capacity_scope", "system_boundary",
    "source_ids", "date_mapped",
]

with open(MAP_A_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=MAP_A_COLS)
    writer.writeheader()
    for i, (entity, le) in enumerate(zip(ue, linked_entities)):
        source_ids = [s["source_id"] for s in le.get("sources", [])]
        writer.writerow({
            "unmapped_index":   i,
            "technology_name":  entity.get("technology_name", ""),
            "linked_entity_id": le["linked_entity_id"],
            "tech_id":          le["tech_id"],
            "process_id":       le["process_id"],
            "geographic_scope": le["scope"].get("geographic_scope", ""),
            "temporal_scope":   le["scope"].get("temporal_scope", ""),
            "capacity_scope":   le["scope"].get("capacity_scope", ""),
            "system_boundary":  le["scope"].get("system_boundary", ""),
            "source_ids":       json.dumps(source_ids),
            "date_mapped":      today,
        })

print(f"Provenance map: saved {len(linked_entities)} rows -> {MAP_A_PATH}")

# --- Entity lookup maps: original name -> assigned ID + resolution status ---
_status = globals().get("resolution_status", {})

for entity_type, cfg in ENTITY_CONFIG.items():
    path       = MAPPING_DIR / f"{entity_type}_map.csv"
    name_field = cfg["name_field"]
    id_field   = cfg["id_field"]
    registry   = load_registry(entity_type)
    name_to_id = {r[name_field]: r[id_field] for r in registry}

    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=[name_field, id_field, "status"])
        writer.writeheader()
        for name, rid in name_to_id.items():
            status = _status.get(entity_type, {}).get(name, "pre-existing")
            writer.writerow({name_field: name, id_field: rid, "status": status})
    print(f"Entity lookup map: {entity_type}_map.csv  ({len(name_to_id)} rows)")

# Attribute lookup map
attr_path = MAPPING_DIR / "attribute_map.csv"
with open(attr_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["attribute_name", "attribute_id", "status"])
    writer.writeheader()
    for name, aid in attr_ids.items():
        writer.writerow({"attribute_name": name, "attribute_id": aid, "status": "created"})
print(f"Entity lookup map: attribute_map.csv  ({len(attr_ids)} rows)")

# Scope lookup maps
for scope_type in SCOPE_CONFIG:
    path = MAPPING_DIR / f"{scope_type}_map.csv"
    scope_entries = {val: token for (st, val), token in scope_ids.items() if st == scope_type}
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["original_value", "scope_token", "status"])
        writer.writeheader()
        for val, token in scope_entries.items():
            writer.writerow({"original_value": val, "scope_token": token, "status": "created"})
    print(f"Entity lookup map: {scope_type}_map.csv  ({len(scope_entries)} rows)")

Provenance map: saved 99 rows -> ..\motel-db\mapping\unmapped_to_linked.csv
Entity lookup map: technology_map.csv  (55 rows)
Entity lookup map: process_map.csv  (39 rows)
Entity lookup map: source_map.csv  (25 rows)
Entity lookup map: carrier_map.csv  (37 rows)
Entity lookup map: attribute_map.csv  (15 rows)
Entity lookup map: geographic_scope_map.csv  (1 rows)
Entity lookup map: temporal_scope_map.csv  (4 rows)
Entity lookup map: capacity_scope_map.csv  (7 rows)
Entity lookup map: system_boundary_map.csv  (3 rows)


In [14]:
# --- Per-entity audit report: joins all resolved mappings and prints a structured summary for each entity ---
audit_indices = [0, 1, 2]
audit_results = generate_audit(ue, attr_ids, indices=audit_indices)

print("=== Per-Entity Audit Report ===")
print(f"Auditing {len(audit_results)} of {len(ue)} entities.")
print("Each entry shows the linked entity ID and how every sub-entity")
print("(technology, process, sources, carriers, scope) was resolved.\n")
for entry in audit_results:
    print(json.dumps(entry, indent=2))

=== Per-Entity Audit Report ===
Auditing 3 of 99 entities.
Each entry shows the linked entity ID and how every sub-entity
(technology, process, sources, carriers, scope) was resolved.

{
  "unmapped_index": 0,
  "technology_name": "NH3_CCGT_El",
  "linked_entity_id": "LE_00001",
  "resolution": {
    "technology": {
      "technology_name": "NH3_CCGT_El",
      "tech_id": "TECH_00001",
      "status": "created"
    },
    "process": {
      "process_name": "CCGT",
      "process_id": "PROC_00001",
      "status": "created"
    },
    "sources": [
      {
        "source_name": "report_power_to_ammonia_2018",
        "source_id": "SRC_00001",
        "status": "created"
      },
      {
        "source_name": "conversion_parametrization",
        "source_id": "SRC_00002",
        "status": "created"
      },
      {
        "source_name": "report_danish_energy_2025",
        "source_id": "SRC_00003",
        "status": "created"
      }
    ],
    "carriers": {
      "inputs": [
        